<a href="https://colab.research.google.com/github/aiyman14/Sch-Mgmt-661/blob/main/W10_Image_Classification_CIFAR_Dataset_using_CNNs_Shared.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SCH-MGMT 661: Applications of AI Models  
**Instructor:** Indika Dissanayake  

---
### Tutorial: Image Classification with CNNs (using Keras Functional Model)

---

### Dataset: CIFAR-10 Dataset  

The CIFAR-10 dataset is a collection of **60,000 color images** across **10 classes**, widely used for training machine learning and computer vision algorithms.

- Each image is:
  - Size: **32 x 32 pixels**
  - Channels: **RGB (3 channels)**

- Dataset Composition:
  - **50,000 training images**
  - **10,000 test images**
  - 10 classes (6,000 images per class)

-  **Classes**:
  0. airplane  
  1. automobile  
  2. bird  
  3. cat  
  4. deer  
  5. dog  
  6. frog  
  7. horse  
  8. ship  
  9. truck

This dataset is small but challenging — the images are low resolution and contain natural variations like backgrounds, angles, and colors.

We'll use it to train a **Convolutional Neural Network (CNN)** that can classify these images.


**Reference:**
Foster, D. (2022). Generative deep learning: Teaching machines to paint, write, compose, and play (2nd ed.). O’Reilly Media
This code is based on:
https://github.com/davidADSP/Generative_Deep_Learning_2nd_Edition/blob/main/notebooks/02_deeplearning/02_cnn/cnn.ipynb

## **Step 1: Load and Explore the dataset**

In [7]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

In [8]:
from tensorflow.keras import datasets, utils, layers, models, losses, optimizers


# Load the dataset
(x_train, y_train), (x_test, y_test) = datasets.cifar10.load_data()

NUM_CLASSES = 10



In [9]:
# Check the shape of the images
print("Training data shape:", x_train.shape)  # (50000, 32, 32, 3)
print("Single image shape:", x_train[0].shape)  # (32, 32, 3)

Training data shape: (50000, 32, 32, 3)
Single image shape: (32, 32, 3)


## **Step 2: Normalize the Data and One-Hot Encode the Labels**

In [10]:
# Normalize pixel values (just like we did with Fashion MNIST)
x_train = x_train / 255.0
x_test = x_test / 255.0




**One-Hot Encoding the Labels**

The CIFAR-10 dataset provides labels as **integers from 0 to 9**, where each number corresponds to a class (e.g., 0 = airplane, 1 = automobile, etc.).

In this notebook, we're converting those integer labels to **one-hot encoded vectors**, like this:

Label = 3 → [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]


Why?

We plan to use the **`categorical_crossentropy`** loss function during model training, which expects labels in this one-hot format.

💡 **Note**:  
In the Fashion-MNIST notebook, we likely used **`sparse_categorical_crossentropy`**, which works directly with integer labels — that's why one-hot encoding wasn't needed there.

So both approaches are valid — they just require matching label formats and loss functions.



In [11]:
# one-hot encode the labels

y_train = utils.to_categorical(y_train, NUM_CLASSES)
y_test = utils.to_categorical(y_test, NUM_CLASSES)


## **Step 3: Model Building (Functional API) and Compilation**

In this section, we define a **Convolutional Neural Network (CNN)** using the **Functional API** in Keras.  
This approach gives us more flexibility compared to `Sequential`.

We're using several advanced layers:

-  **Conv2D**: Convolution layers with 3×3 kernels to learn local patterns.
-  **Strides = 2**: Used instead of pooling to reduce the spatial dimensions (downsampling).
-  **Padding = "same"**: Keeps output dimensions consistent.
-  **BatchNormalization**: Normalizes layer outputs to stabilize and accelerate training.
-  **LeakyReLU**: Activation function that avoids "dead neurons" (improves flow of gradients).
-  **Dropout**: Regularizes the model to prevent overfitting.
-  **Dense + Softmax**: Final layers for classifying into 10 CIFAR categories.

👉 Ex 1: Fill in the missing arguments.

In [ ]:
input_layer = layers.Input(shape=(32, 32, 3))

# Convolution Block 1 (no downsampling yet):
x = layers.Conv2D(filters=32, kernel_size=3, strides=1, padding="same")(input_layer)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 2 (with downsampling):
x = layers.Conv2D(filters=32, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Convolution Block 3
# Use 64 filters, kernel size 3, stride 1, padding "same"
x = layers.Conv2D(
    filters=64,
    kernel_size=3,
    strides=1,
    padding="same"
)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)


# Convolution Block 4
x = layers.Conv2D(filters=64, kernel_size=3, strides=2, padding="same")(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)

# Dense Layers:
x = layers.Flatten()(x)
#x = layers.GlobalAveragePooling2D()(x) # GlobalAveragePooling2D compresses each feature map into a single number — reducing overfitting and making the model lighter and smarter

x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.LeakyReLU()(x)
x = layers.Dropout(rate=0.5)(x)

# output layer
output_layer = layers.Dense(NUM_CLASSES, activation = "softmax")(x)

model = models.Model(input_layer, output_layer)


In [ ]:
model.summary()

**Compiling the Model**

- **Optimizer**: adam (adaptive learning rate; strong default choice)
- **Loss**: categorical_crossentropy (or use 'sparse_categorical_crossentropy' )if not one-hot encoded
- **Metric**: accuracy


👉 Ex 2: Fill in the missing arguments.

In [ ]:
from tensorflow.keras.optimizers import Adam

# Compile the model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## **Step 4: Training and Evaluating the Model**
Now that the model is built and compiled, we can begin **training** — the process where the network learns from the data by adjusting its internal weights to minimize error.

During training, the model iteratively improves its predictions using **forward** and **backward propagation** over several *epochs*.

---

**Key Terms**

- **Epoch:** One complete pass through the entire training dataset.  
- **Batch size:** Number of samples processed before the model updates its weights.  
- **Validation split:** Portion of training data set aside to monitor how well the model generalizes (helps detect overfitting).

---

**Training Parameters:**

- Epochs → `10`  
- Batch size → `32`  
- Validation split → `0.2` (20% of training data used for validation)

---

👉 Ex 3: Fill in the missing arguments.

In [ ]:
# train the model
history = model.fit(
    x_train,
    y_train,
    batch_size=32,
    epochs=10,
    shuffle=True, # ensures batches are randomized each epoch
    validation_split=0.2,
)

**Evaluate the Model**

In [ ]:
# accuracy and loss plots

def plot_training_history(history):
    import matplotlib.pyplot as plt

    plt.figure(figsize=(12, 4))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
    plt.title("Training vs. Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Validation Loss')
    plt.title("Training vs. Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
plot_training_history(history)

In [ ]:
# Final test evaluation (once)
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.3f} | Test loss:  {test_loss:.3f}")

## **Step 5: Make Predictions**

In [ ]:
CLASSES = np.array(
    [
        "airplane",
        "automobile",
        "bird",
        "cat",
        "deer",
        "dog",
        "frog",
        "horse",
        "ship",
        "truck",
    ]
)

preds = model.predict(x_test)
preds_single = CLASSES[np.argmax(preds, axis=-1)]
actual_single = CLASSES[np.argmax(y_test, axis=-1)]

In [ ]:
import matplotlib.pyplot as plt

n_to_show = 10
indices = np.random.choice(range(len(x_test)), n_to_show)

fig = plt.figure(figsize=(15, 3))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

for i, idx in enumerate(indices):
    img = x_test[idx]
    ax = fig.add_subplot(1, n_to_show, i + 1)
    ax.axis("off")
    ax.text(
        0.5,
        -0.35,
        "pred = " + str(preds_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.text(
        0.5,
        -0.7,
        "act = " + str(actual_single[idx]),
        fontsize=10,
        ha="center",
        transform=ax.transAxes,
    )
    ax.imshow(img)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(model.predict(x_test), axis=1)

cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASSES)
fig, ax = plt.subplots(figsize=(12, 10))  # figure size
disp.plot(ax=ax, xticks_rotation=90)
plt.title("Confusion Matrix")
plt.show()